In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


In [2]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.08), ratio=(0.3, 3.3), value='random')
])

transform_test = transforms.Compose([
    
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

full_train_data = datasets.CIFAR10(root="../data", train = True, download=True, transform=transform_test)
train_data = datasets.CIFAR10(root="../data", train = True, download=True, transform=transform_train)
test_data = datasets.CIFAR10(root="../data", train = False, download=True, transform=transform_test)

train_size = int(len(train_data) * .8)
val_size = len(train_data) - train_size

train_subset, val_subset = torch.utils.data.random_split(full_train_data, [train_size, val_size], generator= torch.Generator().manual_seed(123))

train_data = Subset(train_data, train_subset.indices)
val_data = Subset(full_train_data, val_subset.indices)

In [3]:
use_cuda = torch.cuda.is_available()
loader_kwargs = {
    'batch_size': 64,
    'num_workers': 2 if use_cuda else 0,
    'pin_memory': use_cuda
}

if loader_kwargs['num_workers'] > 0:
    loader_kwargs['persistent_workers'] = True

train_loader = DataLoader(train_data, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_data, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_data, shuffle=False, **loader_kwargs)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(f"DataLoader workers: {loader_kwargs['num_workers']}, pin_memory: {loader_kwargs['pin_memory']}")

torch.Size([64, 3, 32, 32])
torch.Size([64])
DataLoader workers: 2, pin_memory: True


In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
CUDA runtime: 12.8


In [5]:
class patch_embedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):  #in channels is 3 for RGB images
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, embed_dim, kernel_size=patch_size, stride=patch_size)
        )

    def forward(self, x):
        x = self.proj(x) #(64, embed_dim, 8, 8) -> (batch_size, embed_dim, num_patches_h, num_patches_w)
        x = x.flatten(2) #flatten the last two dimensions (H and W)
        x = x.transpose(1, 2) #transpose to (batch_size, num_patches, embed_dim)
        # 64 patches of size embed_dim
        return x


In [6]:
class SmallViT(nn.Module):
    def __init__(self,img_size=32, patch_size=4, in_channels=3, embed_dim=192, depth=6, num_heads=6, mlp_dim=768, num_classes=10, droupout=0.1):
        super().__init__()
        self.patch_embed = patch_embedding(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim) 
        #what each patfch looks like after being projected to the embedding dimension

        num_patches = (img_size // patch_size) ** 2 # should be 64 for 32x32 images with 4x4 patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim)) #learnable classification token that summarizes the patch sequence
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim)) #one positional embedding for the CLS token plus each image patch
        self.droupout = nn.Dropout(droupout) #dropout for regularization
        #where each patch is in the image

        #everything so for is for a transformer to adapt to an image input, now we need to add the transformer encoder layers

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, #token size, number of features in each patch embedding
            nhead=num_heads,  
            dim_feedforward=mlp_dim,
            dropout=droupout,
            activation="gelu", #counterpart to relu, smoother and better for training deep networks and transformers
            batch_first=True   #we want our input to be (batch_size, seq_len, embed_dim)
        )

        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=depth) #repeat the encoder layer 'depth' times
        #encoder layer consists of multi-head self attention and a feedforward network, we stack 'depth' of these layers to create the full transformer encoder

        self.norm = nn.LayerNorm(embed_dim) #layer normalization for stability and better training, normalizes the output of the encoder
        self.head = nn.Linear(embed_dim, num_classes) #final linear layer to get class logits(10 classes for CIFAR-10)

    def forward(self, x):
        x = self.patch_embed(x)
        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        x = self.droupout(x)
        x = self.encoder(x)
        x = self.norm(x)  #layer norm for stability and better training, normalizes the output of the encoder
        x = x[:, 0] #use the CLS token as the image-level representation
        x= self.head(x)   #final linear layer to get class logits(10 classes for CIFAR-10)
        return x
    

model = SmallViT().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-3)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Model device: {next(model.parameters()).device}")
print(f"AMP enabled: {use_amp}")
print(model)

Model device: cuda:0
AMP enabled: True
SmallViT(
  (patch_embed): patch_embedding(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (droupout): Dropout(p=0.1, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=128, out_

In [7]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    use_amp = device.type == "cuda"
    amp_device = "cuda" if use_amp else "cpu"

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(amp_device, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy



In [8]:
import copy
import math
epochs = 100
warmup_epochs = 10
warmup_start_factor = 0.2
min_lr_factor = 0.5
decay_epochs = max(1, epochs - warmup_epochs)
log_decay_strength = (1 / min_lr_factor - 1) / math.log1p(decay_epochs)

def warmup_log_lr_lambda(epoch):
    if epoch < warmup_epochs:
        warmup_progress = epoch / max(1, warmup_epochs)
        return warmup_start_factor + (1 - warmup_start_factor) * warmup_progress

    decay_epoch = epoch - warmup_epochs
    return 1 / (1 + log_decay_strength * math.log1p(decay_epoch))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_log_lr_lambda)
best_val_acc = 0.0
best_state = None
patience = 10
epochs_without_improvement = 0
amp_device = "cuda" if use_amp else "cpu"

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(amp_device, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}; best val acc: {best_val_acc:.4f}")
        break


Epoch 1/100 | LR: 0.000100 | Train Loss: 1.9616 | Val Loss: 1.8063 | Val Acc: 0.3475
Epoch 2/100 | LR: 0.000140 | Train Loss: 1.7507 | Val Loss: 1.5963 | Val Acc: 0.4149
Epoch 3/100 | LR: 0.000180 | Train Loss: 1.6096 | Val Loss: 1.4599 | Val Acc: 0.4705
Epoch 4/100 | LR: 0.000220 | Train Loss: 1.5198 | Val Loss: 1.3917 | Val Acc: 0.4910
Epoch 5/100 | LR: 0.000260 | Train Loss: 1.4550 | Val Loss: 1.3582 | Val Acc: 0.5049
Epoch 6/100 | LR: 0.000300 | Train Loss: 1.4003 | Val Loss: 1.2660 | Val Acc: 0.5374
Epoch 7/100 | LR: 0.000340 | Train Loss: 1.3543 | Val Loss: 1.2794 | Val Acc: 0.5403
Epoch 8/100 | LR: 0.000380 | Train Loss: 1.3148 | Val Loss: 1.2112 | Val Acc: 0.5563
Epoch 9/100 | LR: 0.000420 | Train Loss: 1.2809 | Val Loss: 1.1938 | Val Acc: 0.5628
Epoch 10/100 | LR: 0.000460 | Train Loss: 1.2573 | Val Loss: 1.1622 | Val Acc: 0.5750
Epoch 11/100 | LR: 0.000500 | Train Loss: 1.2273 | Val Loss: 1.1218 | Val Acc: 0.5936
Epoch 12/100 | LR: 0.000433 | Train Loss: 1.1642 | Val Loss: 1.

In [9]:
model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 0.7410 | Test Acc: 0.7593
